# 04 · The Magnitude Model

**Purpose.** Notebook 03 predicts *which* state a trip lands in. Three of the four states carry a determined revenue outcome once the state is known:

| State | Revenue |
|---|---|
| `no_purchase` | zero |
| `unchanged` | full coverage: every travelling line, every travelled day |
| `reduces` | **unknown — needs a size** |
| `increases` | **unknown — needs a size** |

A trip that reduces from ten days to nine and one that reduces from ten to two are the same state and nowhere near the same money. This notebook supplies the size, so notebook 05 can compute

$$\mathbb{E}[\text{revenue}] = \sum_{\text{state}} P(\text{state}) \times \text{revenue}(\text{state})$$

with a real number in every term.

**Two models, fitted only on trips in the relevant state**

| Model | Target | Form |
|---|---|---|
| Reduction depth | days bought / days travelled, and lines active / lines travelling | Binomial proportion, weighted by trip length |
| Increase size | extra days bought beyond days travelled | Positive count |

**Same architecture as notebook 03, for the same reason.** Non-price features go to gradient-boosted trees fitted on standard-price trips only; price enters as a sign-constrained parametric term on top. Magnitude must respond to price — a bigger increase should produce a deeper cut, not merely more cutters — and that response has to be smooth across the price range and correctly signed everywhere.

**One contamination to keep in view throughout.** The labels are reconstructed, not observed. Notebook 02 measured the `reduces` rule at roughly 0.72 precision, so about a quarter of the trips this model trains on are incidental one-day shortfalls rather than deliberate reductions. They are shallow by construction and pull the fitted depth toward zero. §5 sizes that bias rather than assuming it away.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.optimize import minimize

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score

DATA = Path("../data-science-portfolio/data/synthetic")


def load(name):
    pq, csv = DATA / f"{name}.parquet", DATA / f"{name}.csv"
    if pq.exists():
        return pd.read_parquet(pq)
    if csv.exists():
        return pd.read_csv(csv)
    raise FileNotFoundError(f"{name} not found in {DATA.resolve()} — run notebooks 01–03 first.")


accounts = load("accounts")
macro = load("macro")
trips = load("trips").merge(accounts, on="account_id", how="left")
trips = trips.merge(macro[["month", "macro_index"]], on="month", how="left")

PRICE_FIRST, PRICE_ADDL = 10.0, 5.0
DAY_THRESH = 0.70                      # the rule notebook 02 selected

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True,
                     "grid.alpha": 0.25, "font.size": 9})
INK, ACCENT, MUTED = "#201e1d", "#1c4ed8", "#9a9694"

print(f"{len(trips):,} trips loaded")

---
## 1 · Labels, features and the split

Identical to notebook 03 so the two models compose cleanly. The label rule, the leakage exclusions, the expanding-window history and the temporal split all carry over unchanged — if they differed, the state probabilities and the magnitudes would describe different populations.

In [ ]:
def reconstruct_label(df, day_thresh=DAY_THRESH):
    ratio = df["days_bought"] / df["days_traveled"]
    lines = df["lines_active"] / df["lines_traveling"].clip(lower=1)
    y = pd.Series("unchanged", index=df.index, dtype=object)
    y[ratio > 1.0] = "increases"
    y[(ratio < day_thresh) | (lines < 0.999)] = "reduces"
    y[df["days_bought"] == 0] = "no_purchase"
    return y


def build_features(df):
    out = pd.DataFrame(index=df.index)
    out["is_business"] = df["is_business"].astype(int)
    out["lines_on_account"] = df["lines_on_account"]
    out["tenure_months"] = df["tenure_months"]
    out["premium"] = (df["plan_tier"] == "premium").astype(int)
    out["value_tier"] = (df["plan_tier"] == "value").astype(int)
    out["lines_traveling"] = df["lines_traveling"]
    out["days_traveled"] = df["days_traveled"]
    for r in ["europe", "latam", "apac", "canada"]:
        out[f"region_{r}"] = (df["region"] == r).astype(int)
    out["month_of_year"] = df["month"] % 12
    out["macro_index"] = df["macro_index"]
    return out


trips["y"] = reconstruct_label(trips)
trips["log_price"] = np.log(trips["price_multiplier"])

hist = trips[["account_id", "month"]].sort_values(["account_id", "month"])
cum = hist.groupby("account_id").cumcount()
same = hist.groupby(["account_id", "month"]).cumcount()
trips["prior_trips"] = (cum - same).reindex(trips.index).fillna(0).astype(int)

X = build_features(trips)
X["prior_trips"] = trips["prior_trips"]

month = trips["month"].to_numpy()
tr, te = month <= 19, month >= 20            # no calibration slice needed here
STD = (trips["price_multiplier"] == 1.0).to_numpy()

print(trips["y"].value_counts().to_string())
print(f"\nreduces  — train {((trips.y=='reduces') & tr).sum():,}   test {((trips.y=='reduces') & te).sum():,}")
print(f"increases — train {((trips.y=='increases') & tr).sum():,}   test {((trips.y=='increases') & te).sum():,}")

---
## 2 · What reduction actually looks like

Before fitting anything, look at the shape of the thing being modelled. Two questions decide the model form:

1. **Is depth continuous or clustered?** If reductions pile up at one or two specific coverage levels, a conditional mean is the wrong summary.
2. **Does depth move with price, or only frequency?** This is the substantive question. If a price rise makes more people cut but does not make the cuts deeper, the revenue arithmetic is far simpler — and the magnitude model matters much less than the state model.

In [ ]:
red = trips[trips.y == "reduces"].copy()
red["day_cov"] = red["days_bought"] / red["days_traveled"]
red["line_cov"] = red["lines_active"] / red["lines_traveling"].clip(lower=1)
red["days_short"] = red["days_traveled"] - red["days_bought"]

inc = trips[trips.y == "increases"].copy()
inc["extra_days"] = inc["days_bought"] - inc["days_traveled"]

print("Reduction depth")
print(red["day_cov"].describe(percentiles=[.1, .25, .5, .75, .9]).round(3).to_string())
print(f"\nlines also cut: {(red.line_cov < 0.999).mean():.1%} of reducing trips")
print(f"\nIncrease size (extra days)")
print(inc["extra_days"].describe(percentiles=[.25, .5, .75, .9]).round(2).to_string())

print("\nDepth by price — does a higher price cut deeper, or just cut more often?")
depth = trips.assign(is_red=(trips.y == "reduces")).groupby("price_multiplier").agg(
    trips=("trip_id", "size"), reduce_rate=("is_red", "mean")).round(4)
depth["mean_coverage_when_reducing"] = red.groupby("price_multiplier")["day_cov"].mean().round(3)
depth["mean_days_short"] = red.groupby("price_multiplier")["days_short"].mean().round(2)
print(depth.to_string())

fig, axes = plt.subplots(1, 3, figsize=(12, 3))
axes[0].hist(red["day_cov"], bins=40, color=ACCENT, edgecolor="white", linewidth=0.3)
axes[0].set_title("coverage when reducing")
axes[0].set_xlabel("days bought / days travelled")

axes[1].hist(inc["extra_days"], bins=range(0, int(inc.extra_days.quantile(0.99)) + 2),
             color=ACCENT, edgecolor="white", linewidth=0.3)
axes[1].set_title("extra days when increasing")
axes[1].set_xlabel("days beyond travel")

axes[2].plot(depth.index, depth["reduce_rate"], marker="o", color=ACCENT,
             linewidth=1.8, label="P(reduce)")
ax2 = axes[2].twinx()
ax2.plot(depth.index, depth["mean_coverage_when_reducing"], marker="s", color=INK,
         linewidth=1.8, label="coverage | reduce")
ax2.grid(False)
axes[2].set_xlabel("price multiplier")
axes[2].set_ylabel("P(reduce)", color=ACCENT)
ax2.set_ylabel("coverage when reducing", color=INK)
axes[2].set_title("frequency vs depth")
plt.tight_layout()
plt.show()

print("\nBoth lines should move with price. If only P(reduce) moves, the magnitude")
print("model is a formality; if coverage falls too, the two effects compound and")
print("a state-only model would understate the damage.")

---
## 3 · Reduction depth

Modelled as a **binomial proportion**: of the `days_traveled` days available, how many get bought. Weighting each trip by its length is what makes this the right form — a ten-day trip carries ten times the revenue information of a one-day trip, and an unweighted mean over ratios would treat them alike.

Two components, because a household can cut days, cut lines, or both:

- **day coverage** — `days_bought / days_traveled`
- **line coverage** — `lines_active / lines_traveling`

The architecture mirrors notebook 03: trees on non-price features fitted on standard-price trips only, then a parametric price term. Fitting the trees on all prices would let them learn the price effect through region and month, exactly the contamination that flattened stage 2 in the previous notebook.

In [ ]:
def logit(p):
    p = np.clip(p, 1e-4, 1 - 1e-4)
    return np.log(p / (1 - p))


def expit(z):
    return 1.0 / (1.0 + np.exp(-z))


BOUND = -5.0      # a coefficient at the wall means unidentified, not strong


def fit_coverage(target_col, denom_col, label, bound=None):
    """Trees on non-price features (standard price only), then a price term."""
    m_red = (trips.y == "reduces").to_numpy()
    fit_rows = m_red & tr & STD
    n = trips[denom_col].to_numpy().clip(1)
    cov = (trips[target_col].to_numpy() / n)

    base = HistGradientBoostingRegressor(
        max_iter=300, learning_rate=0.06, max_leaf_nodes=15, min_samples_leaf=40,
        l2_regularization=1.0, early_stopping=True, validation_fraction=0.15,
        random_state=0,
    ).fit(X[fit_rows], logit(cov[fit_rows]), sample_weight=n[fit_rows])

    # Parametric price term: higher price -> lower coverage, enforced by a bound.
    rows = m_red & tr
    z0 = base.predict(X[rows])
    lp = trips.loc[rows, "log_price"].to_numpy()
    biz = trips.loc[rows, "is_business"].to_numpy().astype(float)
    obs = cov[rows]
    wt = n[rows]

    def loss(th):
        z = z0 + th[0] * lp + th[1] * lp * biz
        p = expit(z)
        return -np.sum(wt * (obs * np.log(np.clip(p, 1e-9, 1)) +
                             (1 - obs) * np.log(np.clip(1 - p, 1e-9, 1)))) / wt.sum()

    lo = BOUND if bound is None else bound
    r = minimize(loss, np.zeros(2), method="L-BFGS-B",
                 bounds=[(lo, 0), (-4, 4)], options={"maxiter": 400})
    theta = r.x
    if abs(theta[0] - lo) < 1e-6:
        print(f"  WARNING: price coefficient pinned at the bound ({lo}). The")
        print("  parameter is not identified — too few trips move this channel.")

    net = theta[0] + theta[1] * biz
    print(f"{label}: price coefficient {theta[0]:+.3f}, business interaction {theta[1]:+.3f}")
    print(f"  net effect min/median/max {net.min():+.3f} / {np.median(net):+.3f} / {net.max():+.3f}"
          f"   wrong sign: {(net > 0).mean():.2%}")
    if (net > 0).mean() > 0.01:
        print("  refitting without the interaction to guarantee the sign")
        r = minimize(loss, np.zeros(2), method="L-BFGS-B",
                     bounds=[(lo, 0), (0, 0)], options={"maxiter": 400})
        theta = r.x
    return base, theta


print("Fitting reduction depth\n")
day_base, day_theta = fit_coverage("days_bought", "days_traveled", "day coverage ")
# Only a small minority of reducing trips cut lines rather than days, so this
# channel carries little information. A tight bound keeps a thin estimate from
# dominating the revenue arithmetic.
line_share = ((trips.y == "reduces") &
              (trips.lines_active < trips.lines_traveling)).sum() / max((trips.y == "reduces").sum(), 1)
print(f"\nline cuts occur on {line_share:.1%} of reducing trips")
line_base, line_theta = fit_coverage("lines_active", "lines_traveling", "line coverage",
                                     bound=-1.5)


def predict_coverage(base, theta, Xf, df, mult=None):
    lp = np.log(float(mult)) * np.ones(len(df)) if mult is not None \
        else df["log_price"].to_numpy()
    z = base.predict(Xf) + theta[0] * lp + theta[1] * lp * df["is_business"].to_numpy().astype(float)
    return np.clip(expit(z), 0.02, 1.0)

In [ ]:
# Out-of-sample accuracy, against the two benchmarks that matter
m_te = ((trips.y == "reduces") & te).to_numpy()
df_te = trips[m_te]
obs_cov = (df_te["days_bought"] / df_te["days_traveled"]).to_numpy()
pred_cov = predict_coverage(day_base, day_theta, X[m_te], df_te)

train_mean = ((trips.loc[(trips.y == "reduces") & tr, "days_bought"] /
               trips.loc[(trips.y == "reduces") & tr, "days_traveled"]).mean())

print("Day coverage, held-out trips that reduced\n")
print(pd.DataFrame({
    "MAE": [mean_absolute_error(obs_cov, pred_cov),
            mean_absolute_error(obs_cov, np.full_like(obs_cov, train_mean))],
    "R2": [r2_score(obs_cov, pred_cov), 0.0],
    "mean predicted": [pred_cov.mean(), train_mean],
}, index=["model", "constant (train mean)"]).round(4).to_string())
print(f"\nobserved mean coverage on test: {obs_cov.mean():.4f}")
print("\nThe model has to beat the constant on MAE. Coverage is a narrow, noisy")
print("quantity, so a modest R2 is expected — the aggregate mean matters more,")
print("since that is what the revenue arithmetic consumes.")

# Does predicted depth move with price, smoothly and in the right direction?
GRID = np.round(np.arange(0.90, 1.2001, 0.025), 4)
curve = [predict_coverage(day_base, day_theta, X[m_te], df_te, m).mean() for m in GRID]
viol = int((np.diff(curve) > 1e-9).sum())
print(f"\nPredicted coverage at 0.90 / 1.00 / 1.20: "
      f"{curve[0]:.4f} / {curve[len(curve)//3]:.4f} / {curve[-1]:.4f}")
print(f"Monotonicity violations across the range: {viol}")

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(GRID, curve, color=ACCENT, linewidth=1.8)
ax.axvline(1.0, color=MUTED, linestyle="--", linewidth=0.9)
ax.set_xlabel("price multiplier")
ax.set_ylabel("predicted day coverage | reduces")
ax.set_title("Do cuts get deeper as price rises?")
plt.tight_layout()
plt.show()

---
## 4 · Increase size

Extra days bought beyond days travelled, conditional on the trip increasing. A positive count with a floor of one, so it is modelled on the log scale and exponentiated back.

The price term carries the opposite sign: a **deeper discount buys more extra days**.

Increases are not confined to discounted trips — some accounts buy beyond their travel days at any price, for convenience or because travel ran longer than planned. Most of the volume sits at the standard rate, so the price term here is estimated off a thin slice at the edges of the range and should be read as weak. If the fitted coefficient comes out at zero, that is a real finding about this population rather than a broken fit.

In [ ]:
m_inc = (trips.y == "increases").to_numpy()
extra = (trips["days_bought"] - trips["days_traveled"]).to_numpy().astype(float)

print("Price levels at which increases occur")
print(trips.loc[m_inc, "price_multiplier"].value_counts().sort_index().to_string())
off_std = trips.loc[m_inc, "price_multiplier"].ne(1.0).mean()
print(f"\n{off_std:.1%} of increases occur away from the standard rate — the price term")
print("is estimated off that slice, so treat it as weakly identified.")

fit_rows = m_inc & tr & STD
if fit_rows.sum() < 50:
    # No increases at the standard rate: fall back to all training increases and say so.
    fit_rows = m_inc & tr
    print("\nToo few increases at the standard rate; the tree is fitted on all")
    print("training increases. Price and baseline are less cleanly separated here.")

inc_base = HistGradientBoostingRegressor(
    max_iter=200, learning_rate=0.06, max_leaf_nodes=15, min_samples_leaf=30,
    l2_regularization=1.0, early_stopping=True, validation_fraction=0.15, random_state=0,
).fit(X[fit_rows], np.log(np.clip(extra[fit_rows], 1, None)))

rows = m_inc & tr
z0 = inc_base.predict(X[rows])
lp = trips.loc[rows, "log_price"].to_numpy()
obs = np.log(np.clip(extra[rows], 1, None))


def inc_loss(th):
    return np.mean((obs - (z0 + th[0] * lp)) ** 2)


inc_theta = minimize(inc_loss, np.zeros(1), method="L-BFGS-B",
                     bounds=[(-15, 0)], options={"maxiter": 300}).x
print(f"\nprice coefficient on log extra days: {inc_theta[0]:+.3f} "
      f"(negative = a deeper discount buys more days)")
if abs(inc_theta[0]) < 1e-6:
    print("Zero: increase size does not respond to price in this population.")
    print("Revenue from increases then scales with price only, not with volume.")


def predict_extra(Xf, df, mult=None):
    lp = np.log(float(mult)) * np.ones(len(df)) if mult is not None \
        else df["log_price"].to_numpy()
    return np.clip(np.exp(inc_base.predict(Xf) + inc_theta[0] * lp), 1.0, 30.0)


m_ite = m_inc & te
if m_ite.sum() > 20:
    pe = predict_extra(X[m_ite], trips[m_ite])
    oe = extra[m_ite]
    print(f"\nHeld-out increases: n={m_ite.sum():,}")
    print(f"  MAE model {mean_absolute_error(oe, pe):.3f}  vs constant "
          f"{mean_absolute_error(oe, np.full_like(oe, extra[rows].mean())):.3f}")
    print(f"  mean predicted {pe.mean():.2f}  observed {oe.mean():.2f}")
else:
    print(f"\nOnly {m_ite.sum()} increases in the test window — not enough to validate.")

---
## 5 · Label contamination: how much does it bias the depth?

The magnitude model trains on trips *labelled* `reduces`, and notebook 02 measured that rule at roughly 0.72 precision. The false positives are incidental one-day shortfalls, which are shallow by construction — so the fitted depth is biased toward zero, and expected revenue is biased **optimistic**.

Because this is synthetic data the size of that bias is measurable. Comparing depth among truly-reducing trips against depth among all labelled ones gives the correction factor, and it is exported so notebook 05 can run the simulation with and without it.

In production this number is unknowable, which is the argument for having built it here.

In [ ]:
truth = trips["state"].replace({"substitutes": "no_purchase", "leaves": "no_purchase"})
labelled = trips.y == "reduces"
truly = labelled & (truth == "reduces")
falsely = labelled & (truth != "reduces")

cov_all = (trips.loc[labelled, "days_bought"] / trips.loc[labelled, "days_traveled"]).mean()
cov_true = (trips.loc[truly, "days_bought"] / trips.loc[truly, "days_traveled"]).mean()

print(pd.DataFrame({
    "trips": [labelled.sum(), truly.sum(), falsely.sum()],
    "mean coverage": [cov_all, cov_true,
                      (trips.loc[falsely, "days_bought"] /
                       trips.loc[falsely, "days_traveled"]).mean() if falsely.sum() else np.nan],
}, index=["labelled reduces", "truly reduces", "false positives"]).round(4).to_string())

print(f"\nPrecision of the label rule here: {truly.sum() / max(labelled.sum(), 1):.3f}")
DEPTH_BIAS = float((1 - cov_true) / max(1 - cov_all, 1e-9))
print(f"True cuts are {DEPTH_BIAS:.2f}x deeper than the labelled average.")
print("\nExported as depth_bias. Notebook 05 runs the simulation at 1.0 and at this")
print("factor; the gap between them is the cost of not observing the state directly.")

---
## 6 · Revenue per state

The pricing identity, applied per state with the predicted magnitudes filled in:

```
revenue = (10 + 5 × (lines_active − 1)) × days_bought × price_multiplier
```

| State | lines | days |
|---|---|---|
| `no_purchase` | 0 | 0 |
| `unchanged` | all travelling | all travelled |
| `reduces` | predicted line coverage | predicted day coverage |
| `increases` | all travelling | travelled + predicted extra |

`revenue_by_state` below is the single function notebook 05 calls. Giving the simulation one interface rather than a set of loose predictions is what keeps the two notebooks from drifting apart.

In [ ]:
CLASSES = ["no_purchase", "reduces", "unchanged", "increases"]


def revenue_by_state(Xf, df, mult=1.0, depth_scale=1.0):
    """Revenue each trip would produce in each state, at price `mult`.

    depth_scale > 1 deepens predicted reductions, for the §5 contamination
    bracket. Returns an (n, 4) array in CLASSES order.
    """
    lines_t = df["lines_traveling"].to_numpy().astype(float)
    days_t = df["days_traveled"].to_numpy().astype(float)

    def rev(lines, days):
        lines = np.maximum(lines, 0)
        return np.where(lines > 0,
                        (PRICE_FIRST + PRICE_ADDL * (lines - 1)) * days * mult, 0.0)

    day_cov = predict_coverage(day_base, day_theta, Xf, df, mult)
    line_cov = predict_coverage(line_base, line_theta, Xf, df, mult)
    day_cov = np.clip(1 - depth_scale * (1 - day_cov), 0.02, 1.0)
    line_cov = np.clip(1 - depth_scale * (1 - line_cov), 0.02, 1.0)

    out = np.zeros((len(df), 4))
    out[:, CLASSES.index("no_purchase")] = 0.0
    out[:, CLASSES.index("unchanged")] = rev(lines_t, days_t)
    out[:, CLASSES.index("reduces")] = rev(np.maximum(np.round(lines_t * line_cov), 1),
                                           np.maximum(np.round(days_t * day_cov), 1))
    out[:, CLASSES.index("increases")] = rev(lines_t, days_t + predict_extra(Xf, df, mult))
    return out


R = revenue_by_state(X[te], trips[te], 1.0)
print("Mean revenue per trip by state, at the standard price\n")
print(pd.Series(R.mean(0), index=CLASSES).round(2).to_string())

print("\nAgainst what actually happened on those trips")
act = trips[te].groupby("y")["pass_revenue"].mean().reindex(CLASSES)
print(pd.DataFrame({"predicted": pd.Series(R.mean(0), index=CLASSES),
                    "actual": act}).round(2).to_string())
print("\nThese should be close for unchanged and reduces. A large gap on reduces")
print("means the depth model is off in a way that flows straight into the revenue.")

In [ ]:
# End-to-end check: does state x magnitude reproduce observed revenue?
scored = load("scored_trips")
sc_te = scored[scored["slice"] == "test"].set_index("trip_id")
te_ids = trips.loc[te, "trip_id"]
common = te_ids[te_ids.isin(sc_te.index)]

if len(common) > 0:
    idx = trips["trip_id"].isin(common).to_numpy() & te
    P = sc_te.loc[trips.loc[idx, "trip_id"], [f"p_{c}" for c in CLASSES]].to_numpy()
    Rv = revenue_by_state(X[idx], trips[idx], 1.0)
    exp_rev = (P * Rv).sum(axis=1)
    obs_rev = trips.loc[idx, "pass_revenue"].to_numpy()

    print(f"Expected vs observed revenue on {idx.sum():,} held-out trips\n")
    print(f"  expected total  ${exp_rev.sum():,.0f}")
    print(f"  observed total  ${obs_rev.sum():,.0f}")
    print(f"  error           {(exp_rev.sum() / obs_rev.sum() - 1):+.2%}")
    print(f"\n  per trip: expected ${exp_rev.mean():.2f}  observed ${obs_rev.mean():.2f}")
    print("\nThis is the number that matters. The simulation's whole output is this")
    print("calculation repeated across prices, so an error here scales into every")
    print("scenario. Within a few percent is the target.")
    REV_ERROR = float(exp_rev.sum() / obs_rev.sum() - 1)
else:
    REV_ERROR = np.nan
    print("scored_trips does not overlap the test window — re-run notebook 03.")

---
## 7 · The revenue curve

State probabilities and magnitudes combined across the price range. This is a preview of notebook 05's answer, without the macro uncertainty, seasonality or departure cost — a deterministic sketch of the shape.

Two things it should show: a smooth curve with no kinks, and whether revenue is still climbing at +20%. If it is, the recommendation will be bounded by leadership's cap rather than by an interior optimum, and that has to be reported as a constraint rather than as an optimisation result.

In [ ]:
sweep = load("price_sweep").set_index("price_multiplier")
base_rev = trips.loc[te, "pass_revenue"].sum()

rows = []
for m in GRID:
    if float(m) in sweep.index:
        p = sweep.loc[float(m), CLASSES].to_numpy()
    else:
        j = np.argmin(np.abs(sweep.index.to_numpy() - m))
        p = sweep.iloc[j][CLASSES].to_numpy()
    Rv = revenue_by_state(X[te], trips[te], m)
    rows.append({"price_multiplier": m, "revenue": float((Rv * p).sum(axis=1).sum())})

curve = pd.DataFrame(rows).set_index("price_multiplier")
curve["vs_standard"] = curve["revenue"] / curve.loc[1.0, "revenue"] - 1
print(curve.round({"revenue": 0, "vs_standard": 4}).to_string())

best = curve["revenue"].idxmax()
print(f"\nRevenue-maximising price in the tested range: {best:.3f}")
if best >= GRID[-1] - 1e-9:
    print("Revenue is still climbing at the +20% ceiling. The cap is binding, so the")
    print("recommendation is a bounded answer: the constraint chose +20%, not the model.")

fig, ax = plt.subplots(figsize=(5.6, 3.2))
ax.plot(curve.index, curve["revenue"] / 1e6, color=ACCENT, linewidth=2)
ax.axvline(1.0, color=MUTED, linestyle="--", linewidth=0.9)
ax.axvline(1.20, color=INK, linestyle=":", linewidth=1.2)
ax.set_xlabel("price multiplier")
ax.set_ylabel("modelled revenue ($M, test window)")
ax.set_title("Revenue across the tested range")
plt.tight_layout()
plt.show()

---
## 8 · Export

In [ ]:
mag = trips[["trip_id", "account_id", "month", "price_multiplier"]].copy()
mag["pred_day_coverage"] = predict_coverage(day_base, day_theta, X, trips)
mag["pred_line_coverage"] = predict_coverage(line_base, line_theta, X, trips)
mag["pred_extra_days"] = predict_extra(X, trips)

# Revenue per state at each price on the grid — the simulation's lookup table
tbl = []
for m in GRID:
    Rv = revenue_by_state(X, trips, m)
    tbl.append(pd.DataFrame({"trip_id": trips["trip_id"].to_numpy(),
                             "price_multiplier": m,
                             **{f"rev_{c}": Rv[:, i] for i, c in enumerate(CLASSES)}}))
rev_tbl = pd.concat(tbl, ignore_index=True)

meta = pd.DataFrame({
    "key": ["depth_bias", "revenue_error", "day_price_coef", "line_price_coef",
            "extra_price_coef"],
    "value": [DEPTH_BIAS, REV_ERROR, float(day_theta[0]), float(line_theta[0]),
              float(inc_theta[0])],
})
print(meta.to_string(index=False))

FMT = "parquet"
try:
    import pyarrow  # noqa: F401
except ImportError:
    try:
        import fastparquet  # noqa: F401
    except ImportError:
        FMT = "csv"
        print("\npyarrow/fastparquet not found, writing CSV instead.")

print()
for df, name in [(mag, "trip_magnitudes"), (rev_tbl, "revenue_by_state"),
                 (curve.reset_index(), "deterministic_revenue_curve"),
                 (meta, "magnitude_meta")]:
    path = DATA / f"{name}.{FMT}"
    df.to_parquet(path, index=False) if FMT == "parquet" else df.to_csv(path, index=False)
    print(f"  {path.name:32s} {len(df):>9,} rows")

---
## What this notebook establishes

| Result | Where | Consequence |
|---|---|---|
| Depth moves with price, not just frequency | §2–3 | A state-only model would understate the damage |
| Coverage model beats the constant | §3 | The magnitude carries real information |
| Depth response smooth and correctly signed | §3 | Simulation curves stay defensible |
| Increase size identified on discounts only | §4 | Stated as a limit, not hidden |
| Label contamination sized | §5 | Exported as a bracket, not assumed away |
| Expected revenue reproduces observed | §6 | The composition is arithmetically sound |
| Revenue still climbing at +20% | §7 | The cap binds — a constraint, not an optimum |

### Carry into notebook 05

1. **`revenue_by_state(X, df, mult, depth_scale)`** pairs with notebook 03's `score_trips`. Expected revenue is the elementwise product summed over states.
2. **Bracket twice, not once.** `response_scale` from notebook 03 §6.2 and `depth_scale` from §5 here are independent sources of optimism, and both push the same direction: the unbracketed answer overstates how safe a price rise is.
3. **`no_purchase` is not free.** The revenue table books it at zero, which is right for the pass but wrong for the relationship. Departures carry account-level value, and notebook 05 has to subtract it using the bootstrapped rate from notebook 03 §10.

### Next

`05_simulation.ipynb` — 12 months forward across the price range, with the macro shock drawn once per iteration and applied to every account, so the revenue distribution has an honest width.